In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from uuid import uuid4
from pathlib import Path
from textwrap import dedent

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema

## Data Loading

In [3]:
df = pd.read_json("../data/corpus.jsonl", lines=True)
df.head()

,id,title,content,published_at,word_count,source_url
0,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...,2009-11-16,231,http://majalah-balebat.blogspot.com/2009/11/wa...
1,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...,2009-12-19,265,http://majalah-balebat.blogspot.com/2009/12/wa...
2,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...,2011-06-12,1255,http://majalah-balebat.blogspot.com/2011/06/wa...
3,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...,2018-11-28,246,http://tipscaras.blogspot.com/2017/02/contoh-a...
4,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g...",2023-07-24,69,https://basasunda.com/puisi-bahasa-sunda


## Synthetic Data Generation using LLMs

In [4]:
client = OpenAI()

In [5]:
SYSTEM_PROMPTS = {
    "BEIR": dedent("""
                    Generate short search query-answer pairs based on the provided document.
                    Each query should resemble a natural search input (as if searching on Google or other search engines).
                   
                    Requirements:
                    - Generate a minimum of 3 queries and a maximum of 15 queries, depending on the length of the input text as you see fit.
                    - The query and the answer must be written in Sundanese.
                    - Ensure the response is concise, accurate, and directly relevant to the document.
                    """).strip(),
    "TRIPLET": dedent("""
                    Create a MSMARCO-like triplet dataset from the given document.
                      
                    Each triplet consists of:
                    1. Query: A short, natural search query based on the document.
                    2. Relevant Passage: A passage from the document that directly answers the query.
                    3. Irrelevant Passage: A passage from the document that does not answer the query but is still related to the topic.

                    Requirements:
                    - Generate a minimum of 3 triplets and a maximum of 15 triplets, depending on the length of the input text as you see fit.
                    - Prefer to create as many unique triplet.
                    - Write all elements (query, relevant passage, irrelevant passage) in Sundanese.
                    - Ensure the query and passages are concise and coherent.
                    - Each passage must be self-contained and self-explanatory with context included.
                    """).strip(),
}

### Synthetic BEIR

In [6]:
class BEIRQueryItem(BaseModel):
    search_term: str
    answer: str


class BEIRData(BaseModel):
    queries: list[BEIRQueryItem]

In [7]:
completion_beir = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRData,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["BEIR"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 2],
        },
    ],
)

In [8]:
print(completion_beir)

ParsedChatCompletion[BEIRData](
    id='chatcmpl-BVHMP9Zkd3drW3C2tAaqybysdMNlD',
    choices=[
        ParsedChoice[BEIRData](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRData](
                content='{"queries":[{"search_term":"festival budaya di Bogor","answer":"Festival budaya ieu 
diayakeun di hotél Salak, Bogor."},{"search_term":"tujuan festival Holland Indonésia","answer":"Tujuan festival 
nyaéta pikeun ngenalkeun budaya Walanda ka masyarakat Indonésia."},{"search_term":"siapa wakil duta besar 
Walanda","answer":"Wakil duta besar Walanda nyaéta Annemieke Ruigrok."},{"search_term":"apa anu dipamerkeun dina 
festival","answer":"Dina festival, dipamerkeun batik jeung seni angklung."},{"search_term":"ka hubungan Indonésia 
jeung Walanda","answer":"Hubungan Indonésia jeung Walanda masih keneh keneh sanaos gaduh sajarah anu kurang 
alus."},{"search_term":"ka salapan festival ieu","answer":"Festival budaya ieu ngan diayakeun 
sapoé."},{"search_term":"seni budaya nu dipikaresep Walanda","answer":"Warga Walanda mikaresep seni budaya sareng 
masakan ti Indonésia."},{"search_term":"tempat festival Holland Indonésia","answer":"Festival dilaksanakeun di 
hotél Salak, Jalan Ir Jonda, Kota Bogor."},{"search_term":"organisasi nu ngebantu festival","answer":"Festival 
disokong ku BHI jeung Fined."},{"search_term":"seni batik tulis","answer":"Festival ngenalkeun cara nyieun batik 
tulis."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRData(
                    queries=[
                        BEIRQueryItem(
                            search_term='festival budaya di Bogor',
                            answer='Festival budaya ieu diayakeun di hotél Salak, Bogor.'
                        ),
                        BEIRQueryItem(
                            search_term='tujuan festival Holland Indonésia',
                            answer='Tujuan festival nyaéta pikeun ngenalkeun budaya Walanda ka masyarakat 
Indonésia.'
                        ),
                        BEIRQueryItem(
                            search_term='siapa wakil duta besar Walanda',
                            answer='Wakil duta besar Walanda nyaéta Annemieke Ruigrok.'
                        ),
                        BEIRQueryItem(
                            search_term='apa anu dipamerkeun dina festival',
                            answer='Dina festival, dipamerkeun batik jeung seni angklung.'
                        ),
                        BEIRQueryItem(
                            search_term='ka hubungan Indonésia jeung Walanda',
                            answer='Hubungan Indonésia jeung Walanda masih keneh keneh sanaos gaduh sajarah anu 
kurang alus.'
                        ),
                        BEIRQueryItem(
                            search_term='ka salapan festival ieu',
                            answer='Festival budaya ieu ngan diayakeun sapoé.'
                        ),
                        BEIRQueryItem(
                            search_term='seni budaya nu dipikaresep Walanda',
                            answer='Warga Walanda mikaresep seni budaya sareng masakan ti Indonésia.'
                        ),
                        BEIRQueryItem(
                            search_term='tempat festival Holland Indonésia',
                            answer='Festival dilaksanakeun di hotél Salak, Jalan Ir Jonda, Kota Bogor.'
                        ),
                        BEIRQueryItem(
                            search_term='organisasi nu ngebantu festival',
                            answer='Festival disokong ku BHI jeung Fined.'
                        ),
                        BEIRQueryItem(
                            search_term='seni batik tulis',
                            answer='Fes

### Synthetic Triplet

In [9]:
class TripletItem(BaseModel):
    query: str
    relevant_passage: str
    irrelevant_passage: str


class TripetData(BaseModel):
    triplets: list[TripletItem]

In [10]:
completion_triplet = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=TripetData,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["TRIPLET"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 2],
        },
    ],
)

In [11]:
print(completion_triplet)

ParsedChatCompletion[TripetData](
    id='chatcmpl-BVHMWNcsJGaCBaPWNprwtFMNxU5QM',
    choices=[
        ParsedChoice[TripetData](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[TripetData](
                content='{"triplets":[{"query":"Naon tujuan festival budaya ieu?","relevant_passage":"Festival anu 
boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat.","irrelevant_passage":"Festival 
budaya ieu ngan gel sapoé."},{"query":"Saha anu ngadatangkeun festival di Bogor?","relevant_passage":"Festival 
budaya ieu dihadiran ku wakil duta besar walanda keur indonésé, annemieke ruigrok.","irrelevant_passage":"Sagala 
rupa budaya milik bangsa indonésé dihadirkan dina festival ieu."},{"query":"Kumaha hubungan antara Indonésia sareng
Walanda?","relevant_passage":"Sanajan walanda boga sajarah mangsa ka tukang anu kurang alus di mata masarakat 
indonésé, tapi henteu ngaleungitkeun hubungan eta.","irrelevant_passage":"Urang walanda pohara resep ku masak urang
indonésé."},{"query":"Naon waé budaya anu dipamerkeun di festival?","relevant_passage":"Sagala rupa budaya milik 
bangsa indonésé dihadirkan dina festival ieu, antara lain batik jeung seni angklung.","irrelevant_passage":"Kuring 
gumbira aya di indonésé nu jalma saroméah."},{"query":"Saha wakil anu nyarios di 
festival?","relevant_passage":"Wakil duta besar walanda, annemieke ruigrok kaku pohara gumbira ku lumangsungna 
acara ieu.","irrelevant_passage":"Nurutkeun r. ay. suni wijogawati laku wakil pupuhu fined."},{"query":"Dina taun 
iraha festival ieu dilaksanakeun?","relevant_passage":"Festival ieu dilaksanakeun di hotél salak, jalan ir jonda, 
kota bogor, (15/11/2009).","irrelevant_passage":"Kagétan rupa ogé geus laksana di kota-kota séjén."},{"query":"Naon
hasil anu dipiharep tina festival ieu?","relevant_passage":"Tujonana mung wungkul pikeun ngaronjatkeun hubungan 
antara indonésé jeung walanda.","irrelevant_passage":"Festival budaya ieu ngan gel sapoé."},{"query":"Kumaha eusi 
ucapan annemieke ruigrok?","relevant_passage":"“Indonésé jeung walanda boga hubungan lit,“ ceuk 
manéhna.","irrelevant_passage":"Nurutkeun r. ay. suni wijogawati laku wakil pupuhu fined."},{"query":"Saha anu 
hadir sareng nyarita di festival?","relevant_passage":"Annemieke ruigrok ujar, \'Kuring taji kasoméahanana rahayat 
indonésé.\'","irrelevant_passage":"Festival budaya ieu ngan gel sapoé."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=TripetData(
                    triplets=[
                        TripletItem(
                            query='Naon tujuan festival budaya ieu?',
                            relevant_passage='Festival anu boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do 
nagara ieu ka masarakat.',
                            irrelevant_passage='Festival budaya ieu ngan gel sapoé.'
                        ),
                        TripletItem(
                            query='Saha anu ngadatangkeun festival di Bogor?',
                            relevant_passage='Festival budaya ieu dihadiran ku wakil duta besar walanda keur 
indonésé, annemieke ruigrok.',
                            irrelevant_passage='Sagala rupa budaya milik bangsa indonésé dihadirkan dina festival 
ieu.'
                        ),
                        TripletItem(
                            query='Kumaha hubungan antara Indonésia sareng Walanda?',
                            relevant_passage='Sanajan walanda boga sajarah mangsa ka tukang anu kurang alus di mata
masarakat indonésé, tapi henteu ngaleungitkeun hubungan eta.',
                            irrelevant_passage='Urang walanda pohara resep ku masak urang indonésé.'
                        ),
                        TripletItem(
                            query='Naon waé budaya anu dipamerkeun

## Generate OpenAI Batch Request

### Batch Request Generator

In [12]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel):
    for row in df.itertuples():
        custom_id = str(uuid4())
        job_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": row.content},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": base_model.__name__,
                        "strict": True,
                        "schema": to_strict_json_schema(base_model),
                    },
                },
            },
        }
        
        yield (custom_id, row.id, job_data)

In [13]:
next(generate_batch(df, SYSTEM_PROMPTS["TRIPLET"], TripetData))

('3dc142cf-2407-4a2e-87d9-0835121dda9b',
 '1bcfc529-9788-4f8b-a8e4-c2780922cb9e',
 {'custom_id': '3dc142cf-2407-4a2e-87d9-0835121dda9b',
  'method': 'POST',
  'url': '/v1/chat/completions',
  'body': {'model': 'gpt-4o-mini',
   'messages': [{'role': 'system',
     'content': 'Create a MSMARCO-like triplet dataset from the given document.\n\nEach triplet consists of:\n1. Query: A short, natural search query based on the document.\n2. Relevant Passage: A passage from the document that directly answers the query.\n3. Irrelevant Passage: A passage from the document that does not answer the query but is still related to the topic.\n\nRequirements:\n- Generate a minimum of 3 triplets and a maximum of 15 triplets, depending on the length of the input text as you see fit.\n- Prefer to create as many unique triplet.\n- Write all elements (query, relevant passage, irrelevant passage) in Sundanese.\n- Ensure the query and passages are concise and coherent.\n- Each passage must be self-contained a

In [14]:
def persist_batch(kind: str, schema: BaseModel):
    batch_map_path = Path(f"../data/llm-gen/{kind}/{kind}_map.jsonl")
    batch_req_path = Path(f"../data/llm-gen/{kind}/{kind}_batch.jsonl")
    batch_req_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(batch_req_path, "w") as fm, open(batch_map_path, "w") as mm:
        batch_iter = generate_batch(df, SYSTEM_PROMPTS[kind.upper()], schema)
        for custom_id, doc_id, req in batch_iter:
            json.dump(req, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_id": doc_id}, mm)
            mm.write("\n")
    
    return batch_req_path, batch_map_path

In [15]:
beir_req_path, beir_map_path = persist_batch("beir", BEIRData)
triplet_req_path, triplet_map_path = persist_batch("triplet", TripetData)

### Submit Batch Requests

In [16]:
def submit_batch(path):
    batch_file = client.files.create(file=open(path, "rb"), purpose="batch")

    return client.batches.create(
        input_file_id=batch_file.id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h"
    )

In [17]:
triplet_batch = submit_batch(triplet_req_path.resolve())
print(triplet_batch)

Batch(
    id='batch_681df8a6dd208190b5f10ed38d14f958',
    completion_window='24h',
    created_at=1746794662,
    endpoint='/v1/chat/completions',
    input_file_id='file-3ZSqqULYrT9kLkWXHHvTpS',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746881062,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [18]:
beir_batch = submit_batch(beir_req_path.resolve())
print(beir_batch)

Batch(
    id='batch_681df8c0b5688190b3aeb2a3cb9be8d8',
    completion_window='24h',
    created_at=1746794688,
    endpoint='/v1/chat/completions',
    input_file_id='file-9Dktyi997dNnEs6ACoo52f',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746881088,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)